# AIGC Detector — Training on Kaggle

Kaggle alternative to `notebooks/train_colab.ipynb` — same `train.py`, different environment. Use this if Colab's GPU quota is exhausted; Kaggle's 30hrs/week GPU quota is a completely separate pool.

**Before running:**
1. Notebook Settings (right sidebar) → **Accelerator** → GPU T4 x2 (or P100). If these are greyed out, verify your phone number first: Account Settings → Phone Verification.
2. Notebook Settings → **Internet** → **On** (needed for `git clone`, `pip install`, and the CLIP weight download — off by default on Kaggle).

**Key difference from Colab: no Google Drive.** Kaggle's persistent storage model is different:
- `/kaggle/working/` is writable and persists for the life of this session, and becomes this notebook version's **Output** when you click "Save Version" — downloadable from there afterward.
- There's no direct equivalent of Colab's Drive-symlink trick for surviving a session restart mid-run. To resume across sessions, you need to get your checkpoint file onto Kaggle as a **Dataset** (see step 5) and re-attach it each time — more manual than Colab, but Kaggle sessions also tend to be more stable/less prone to random disconnects than Colab's free tier.
- `evaluate.py`'s own resumability (skips completed conditions) still works exactly the same here — that part doesn't depend on Drive at all, just on `results/robustness_table.csv`/`predictions.csv` existing in `/kaggle/working/`.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none — check Settings > Accelerator")

## 1. Set secrets

`train.py` only needs `HF_TOKEN` (CLIP backbone download — works without it, just rate-limited). Add it via Kaggle's Secrets: Add-ons menu (top toolbar) → Secrets → **Add a new secret**, label it `HF_TOKEN`, then attach it to this notebook and enable it below.

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set.")
except Exception as e:
    print("No HF_TOKEN secret configured — continuing without one (downloads will be rate-limited, not blocked).")

## 2. Clone the repo

Requires Internet ON (Notebook Settings, see intro). Pulls whatever is currently pushed to GitHub — push your local commits first.

In [ ]:
%cd /kaggle/working
!test -d /kaggle/working/choochoo || git clone https://github.com/windyheng/choochoo.git /kaggle/working/choochoo
%cd /kaggle/working/choochoo

## 3. Install dependencies

In [ ]:
%cd /kaggle/working/choochoo
!pip install -q -r requirements.txt

## 4. Reduce num_workers

Kaggle's free GPU sessions typically give you fewer CPUs than `configs/train.yaml`'s `num_workers: 4` assumes (same issue as Colab free tier) — lower it to avoid DataLoader slowness/instability warnings.

In [ ]:
%cd /kaggle/working/choochoo
!sed -i 's/num_workers: 4/num_workers: 2/' configs/train.yaml
!grep num_workers configs/train.yaml

## 5. Resume from an existing checkpoint (optional)

Skip this section entirely for a fresh training run. To continue from a checkpoint trained elsewhere (e.g. Colab):

1. Download the `.pt` file from wherever it currently lives (e.g. your Google Drive `choochoo_checkpoints` folder — right-click the file → Download).
2. On Kaggle: **+ Add Input** (right sidebar) → **New Dataset** → upload that `.pt` file → create the dataset.
3. **Get the exact path** — don't guess it. In the right sidebar's Input section, expand your newly-added dataset, hover over the file, and click the copy icon to get its exact path (Kaggle sometimes nests uploaded files one folder deeper than the dataset name, e.g. `/kaggle/input/choochoo-checkpoints/choochoo-checkpoints/ckpt_step_....pt`, which the copy icon sidesteps entirely).
4. Paste that path's *directory* (not the specific file) into `CHECKPOINT_DATASET_PATH` below, then run the cell — it copies every `.pt` file in that directory into the local `checkpoints/` folder so `train.py`'s `resume_from_latest: true` picks up the highest step number automatically.

In [ ]:
CHECKPOINT_DATASET_PATH = "<paste-the-directory-path-from-step-3-here>"

%cd /kaggle/working/choochoo
!mkdir -p checkpoints
!find {CHECKPOINT_DATASET_PATH} -name "*.pt" -exec cp {{}} checkpoints/ \;
!ls -la checkpoints

## 6. Train

Within this session, re-running this same cell after an interruption resumes automatically (same `resume_from_latest` mechanism as always). If the *session itself* ends (Kaggle sessions run up to ~9-12h, shorter than Colab's cap), `/kaggle/working/checkpoints/` survives until you close/reset the session, but to keep it across a genuinely new session, click **Save Version** periodically (Quick Save) so the checkpoints are preserved in that version's Output, or repeat step 5's upload-as-Dataset flow with the newer checkpoint.

In [ ]:
%cd /kaggle/working/choochoo
!python train.py --config configs/train.yaml

## 7. Evaluate the trained checkpoint

Same resumable `evaluate.py` as the Colab notebook — this part doesn't care about Drive vs. Kaggle storage, it just needs `results/robustness_table.csv`/`predictions.csv` to persist within the session, which `/kaggle/working/` does.

In [ ]:
%cd /kaggle/working/choochoo
from train import find_latest_checkpoint

ckpt = find_latest_checkpoint("checkpoints")
assert ckpt is not None, "no checkpoint found under checkpoints/ — run the train cell first, or step 5 to import one"
print("Using checkpoint:", ckpt)

### 7a. Smoke test first

Proves the pipeline works end-to-end on a tiny subset before committing to the full multi-hour run.

In [ ]:
%cd /kaggle/working/choochoo
!head -50 data/cache/splits/test.csv > /tmp/test_small.csv
!rm -f /tmp/robustness_table_smoke.csv /tmp/predictions_smoke.csv
!python evaluate.py --checkpoint {ckpt} --eval_csv /tmp/test_small.csv --out /tmp/robustness_table_smoke.csv --predictions_out /tmp/predictions_smoke.csv
!cat /tmp/robustness_table_smoke.csv

### 7b. Full run

Only run once the smoke test succeeds. Resumable — re-running this same cell after an interruption (within the same session) skips conditions already in `results/robustness_table.csv`. If you resumed from a checkpoint trained further since this file was last written, delete `results/robustness_table.csv`/`results/predictions.csv` first to force a genuinely fresh run.

In [ ]:
%cd /kaggle/working/choochoo
!python evaluate.py --checkpoint {ckpt} --eval_csv data/cache/splits/test.csv --out results/robustness_table.csv

## 8. Error analysis

In [ ]:
%cd /kaggle/working/choochoo
!python error_analysis.py --predictions results/predictions.csv --out results/error_analysis/

## 9. Required deliverable: infer.py smoke test

In [ ]:
%cd /kaggle/working/choochoo
!mkdir -p /tmp/infer_smoke_test
!find data/raw -type f -name '*.jpg' | head -5 | xargs -I{} cp {} /tmp/infer_smoke_test/
!python infer.py --input_dir /tmp/infer_smoke_test --out results/smoke_test_preds.json --checkpoint {ckpt}
!cat results/smoke_test_preds.json

## 10. Save your results

Click **Save Version** (top right) — this snapshots `/kaggle/working/` (including `checkpoints/` and `results/`) as this notebook version's Output, downloadable from the Output tab afterward. Do this before your session runs out, not just at the very end — a Kaggle session ending unsaved loses anything not yet committed to a version, unlike Colab+Drive where results synced continuously.